# 03 — SFLLD Final Missing-Value Decision Table (Updated BLS Fallback)

**Purpose:** freeze missing-value decisions for the **new dataset produced after completing both Regular-PIT and Strict/LS BLS state fallback**.

This notebook intentionally performs **no imputation and no feature engineering**.

Compared with the previous missing-value decision notebook, this version:

1. loads `SFLLD_modeling_dataset_complete_regular_and_strict_BLS_state_fallback.parquet`;
2. does **not** assume the old 91-column schema;
3. rebuilds missingness and identical-mask groups from the new dataset;
4. separates **Regular-PIT BLS benchmark lineage** from **Strict/LS production lineage**;
5. treats raw MSA/state/fallback components as lineage/intermediate fields when a downstream final field exists;
6. keeps Strict/LS final-resolved macro variables as production macro candidates;
7. keeps Regular-PIT final-resolved variables as **leakage-sensitivity benchmark candidates**, not production predictors;
8. carries forward the loan-level decisions from Notebook 02:
   - `Original_DTI`: keep + missing indicator + Train-only median later;
   - `Property_Valuation_Method`: drop from primary model;
   - `Special_Eligibility_Program`: explicit missing category candidate / later ablation;
9. exports a new machine-readable manifest for the downstream preprocessing/modeling notebook.

**Important:** this notebook freezes *decisions*. It does not fill missing values.

In [1]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 240)

FILE_NAME = "SFLLD_modeling_dataset_complete_regular_and_strict_BLS_state_fallback.parquet"

# Robust to running from either project root or notebooks/.
candidates = [
    Path("data/output/final_modeling_dataset") / FILE_NAME,
    Path("../data/output/final_modeling_dataset") / FILE_NAME,
    Path(FILE_NAME),
]

if "df" in globals() and isinstance(df, pd.DataFrame):
    print("Using existing DataFrame: df")
    DATA_PATH = None
else:
    existing = [p for p in candidates if p.exists()]
    if not existing:
        print("Current working directory:", Path.cwd().resolve())
        print("Checked:")
        for p in candidates:
            print(" -", p.resolve())
        raise FileNotFoundError(
            f"Could not find {FILE_NAME}. "
            "Run from the project root or notebooks directory, or preload a DataFrame named df."
        )
    DATA_PATH = existing[0].resolve()
    print("Loading:", DATA_PATH)
    df = pd.read_parquet(DATA_PATH)

print("Shape:", df.shape)

# Row count and key integrity are fixed; column count is intentionally dynamic
# because the new BLS completion stage adds columns beyond the old 91-column artifact.
assert len(df) == 713000, f"Expected 713000 rows, got {len(df)}"
for required in ["Loan_ID", "Target_24M", "Split", "Dataset_Vintage"]:
    assert required in df.columns, f"Required column missing: {required}"

assert df["Loan_ID"].nunique(dropna=False) == len(df), "Loan_ID must remain unique."
print("Column count detected dynamically:", df.shape[1])

Loading: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_dataset_complete_regular_and_strict_BLS_state_fallback.parquet
Shape: (713000, 105)
Column count detected dynamically: 105


## 1. Immutable baseline fingerprint

In [2]:
BASELINE_ROWS, BASELINE_COLS = df.shape

def stable_series_digest(s):
    h = pd.util.hash_pandas_object(s, index=False).to_numpy(dtype="uint64", copy=True)
    h.sort()
    return hashlib.sha256(h.tobytes()).hexdigest()

baseline = {
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "loan_id_nunique": int(df["Loan_ID"].nunique(dropna=False)),
    "loan_id_duplicate_rows": int(df.duplicated("Loan_ID", keep=False).sum()),
    "loan_id_multiset_sha256": stable_series_digest(df["Loan_ID"]),
    "target_multiset_sha256": stable_series_digest(df["Target_24M"]),
    "split_multiset_sha256": stable_series_digest(df["Split"]),
}
display(pd.DataFrame([baseline]))

,rows,columns,loan_id_nunique,loan_id_duplicate_rows,loan_id_multiset_sha256,target_multiset_sha256,split_multiset_sha256
0,713000,105,713000,0,829486a7e6f634f795956e4bd5c5c47aab69a521a27ebf73ce1704a79a863f19,8c3816c9a9ec9d9094222e2a9dd78407d88edfd5f3d1a6d1346cc0ed88c7e14e,7aa32b3dc7fd870b1804255a6ae8aa691f1d0823fbacf97ab476888c62958978


## 2. Rebuild the complete missingness inventory

In [3]:
inventory = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "non_null_count": [int(df[c].notna().sum()) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [float(df[c].isna().mean() * 100) for c in df.columns],
    "unique_count": [int(df[c].nunique(dropna=True)) for c in df.columns],
})
inventory["has_missing"] = inventory["missing_count"].gt(0)

print("Columns:", len(inventory))
print("Columns with missing:", int(inventory["has_missing"].sum()))
display(inventory.sort_values(["missing_pct", "column"], ascending=[False, True]))

Columns: 105
Columns with missing: 63


,column,dtype,non_null_count,missing_count,missing_pct,unique_count,has_missing
26,Special_Eligibility_Program,str,29515,683485,95.860449,3,True
28,Property_Valuation_Method,string,225680,487320,68.347826,3,True
43,FHFA_MSA_HPI,float64,564870,148130,20.775596,14771,True
44,FHFA_MSA_HPI_Growth_12M,float64,564870,148130,20.775596,23896,True
64,LS_FHFA_MSA_HPI,float64,564870,148130,20.775596,14547,True
65,LS_FHFA_MSA_HPI_Growth_12M,float64,564870,148130,20.775596,23831,True
71,LS_BLS_Employment,float64,584314,128686,18.048527,55872,True
73,LS_BLS_Employment_Growth_12M,float64,584314,128686,18.048527,60620,True
72,LS_BLS_Labor_Force,float64,584314,128686,18.048527,56090,True
70,LS_BLS_Unemployment_Rate,float64,584314,128686,18.048527,260,True


## 3. Collapse identical missing masks

In [4]:
def mask_digest(s):
    arr = s.isna().to_numpy(dtype=np.uint8)
    return hashlib.sha256(arr.tobytes()).hexdigest()

mask_rows = []
for c in df.columns:
    if df[c].isna().any():
        mask_rows.append({
            "column": c,
            "missing_count": int(df[c].isna().sum()),
            "missing_pct": float(df[c].isna().mean() * 100),
            "mask_sha256": mask_digest(df[c]),
        })

mask_df = pd.DataFrame(mask_rows)

if len(mask_df):
    mask_group_table = (
        mask_df.groupby("mask_sha256", as_index=False)
        .agg(
            missing_count=("missing_count", "first"),
            missing_pct=("missing_pct", "first"),
            n_columns=("column", "size"),
            columns=("column", lambda x: " | ".join(x))
        )
        .sort_values(["missing_count", "n_columns"], ascending=[False, False])
        .reset_index(drop=True)
    )
    mask_group_table.insert(0, "mask_group", np.arange(1, len(mask_group_table) + 1))
else:
    mask_group_table = pd.DataFrame(
        columns=["mask_group","mask_sha256","missing_count","missing_pct","n_columns","columns"]
    )

display(mask_group_table)
print("Unique non-empty missing masks:", len(mask_group_table))

,mask_group,mask_sha256,missing_count,missing_pct,n_columns,columns
0,1,aaa46868d3df97e9bfd072a872f04d409bfe62d13f7e90a2aaff648b683d7c36,683485,95.860449,1,Special_Eligibility_Program
1,2,ea86295270e5cff9964ee0479552e0293b123946d6c856b53c2425a05353da81,487320,68.347826,1,Property_Valuation_Method
2,3,7c4703860a72af92a8d522227a1def38260bec4260297e07a120a6d0ba734c5e,148130,20.775596,2,LS_FHFA_MSA_HPI | LS_FHFA_MSA_HPI_Growth_12M
3,4,be283716e3a9771ba5030a2da7dfeac8efb38b9ca15df3cfe8e3729185e4251b,148130,20.775596,2,FHFA_MSA_HPI | FHFA_MSA_HPI_Growth_12M
4,5,d3894e9fe0a3a21b58810bf9d2b9ae97d478fd9fb73a9ccf3944306367d9725a,128686,18.048527,6,LS_BLS_Unemployment_Rate | LS_BLS_Employment | LS_BLS_Labor_Force | LS_BLS_Employment_Growth_12M | LS_BLS_Unemployment_Rate_Change_12M | LS_Labor_Geography_...
5,6,b24f4985de9de6932740db8033eec27edc84818bc143df3edf76664395454910,128657,18.044460,6,BLS_Unemployment_Rate | BLS_Employment | BLS_Labor_Force | BLS_Employment_Growth_12M | BLS_Unemployment_Rate_Change_12M | Labor_Geography_Source
6,7,d232c2a986e5e113940ea4dc279a58b7a9407f591ca55576104a732c4d552274,89555,12.560309,3,MSA | Freddie_MSA | LS_Freddie_MSA
7,8,a783feb24428a6fc665ed19055139b7dff7a3b0024bb62a1a61fee07825756ce,78026,10.943338,1,Original_DTI
8,9,e8f056a7c9226388d93d59691f882702cdf61297283ebaa1cc8008873510e6f1,440,0.061711,3,BLS_Unemployment_Rate_State_Fallback | BLS_Employment_Growth_12M_State_Fallback | BLS_Unemployment_Rate_Change_12M_State_Fallback
9,10,c6554bbdebae74c95be3d8f502cfe71a21d1c7c5bb98bc18d7ebb9afa0a3da06,412,0.057784,3,BLS_Unemployment_Rate_Final | BLS_Employment_Growth_12M_Final | BLS_Unemployment_Rate_Change_12M_Final


Unique non-empty missing masks: 21


## 4. Detect the new BLS lineage

The new dataset may contain more columns than the old 91-column artifact.  
Rather than hard-coding an assumed new width, this cell inventories every BLS-related field and detects final/resolved/fallback fields by name.

The intended modeling interpretation is:

- **Regular PIT BLS final-resolved fields** → benchmark / leakage-sensitivity experiment only.
- **Strict/LS BLS final-resolved fields** → production macro candidates.
- raw local/MSA BLS, state source fields, and fallback construction fields → lineage/intermediate, not independently imputed.

In [5]:
bls_cols = [c for c in df.columns if "BLS" in c or "Labor_Geography" in c]
bls_inventory = inventory[inventory["column"].isin(bls_cols)].copy()

def bls_stage(c):
    u = c.upper()
    strict = u.startswith("LS_") or "STRICT" in u
    regular = not strict

    if "FINAL" in u:
        stage = "FINAL_RESOLVED"
    elif "FALLBACK" in u:
        stage = "FALLBACK_COMPONENT"
    elif "_STATE_" in u or u.startswith("_STATE"):
        stage = "STATE_COMPONENT"
    elif c in {"BLS_Unemployment_Rate","BLS_Employment","BLS_Labor_Force",
               "BLS_Employment_Growth_12M","BLS_Unemployment_Rate_Change_12M",
               "LS_BLS_Unemployment_Rate","LS_BLS_Employment","LS_BLS_Labor_Force",
               "LS_BLS_Employment_Growth_12M","LS_BLS_Unemployment_Rate_Change_12M"}:
        stage = "LOCAL_RAW"
    elif "GEOGRAPHY_SOURCE" in u or "FALLBACK_LEVEL" in u or "APPLIED" in u or "REMAINING_MISSING_REASON" in u:
        stage = "PROVENANCE"
    else:
        stage = "OTHER_AUDIT"

    timeline = "STRICT_LS" if strict else "REGULAR_PIT"
    return timeline, stage

if len(bls_inventory):
    tmp = bls_inventory["column"].apply(bls_stage)
    bls_inventory["timeline"] = [x[0] for x in tmp]
    bls_inventory["lineage_stage"] = [x[1] for x in tmp]

display(bls_inventory.sort_values(["timeline","lineage_stage","column"]))

,column,dtype,non_null_count,missing_count,missing_pct,unique_count,has_missing,timeline,lineage_stage
92,BLS_Employment_Growth_12M_State_Fallback,float64,712560,440,0.061711,9603,True,REGULAR_PIT,FALLBACK_COMPONENT
98,BLS_Fallback_Level,Int8,713000,0,0.000000,3,False,REGULAR_PIT,FALLBACK_COMPONENT
97,BLS_State_Fallback_Applied,Int8,713000,0,0.000000,2,False,REGULAR_PIT,FALLBACK_COMPONENT
93,BLS_Unemployment_Rate_Change_12M_State_Fallback,float64,712560,440,0.061711,528,True,REGULAR_PIT,FALLBACK_COMPONENT
91,BLS_Unemployment_Rate_State_Fallback,float64,712560,440,0.061711,151,True,REGULAR_PIT,FALLBACK_COMPONENT
95,BLS_Employment_Growth_12M_Final,float64,712588,412,0.057784,69478,True,REGULAR_PIT,FINAL_RESOLVED
99,BLS_Geography_Source_Final,string,713000,0,0.000000,3,False,REGULAR_PIT,FINAL_RESOLVED
96,BLS_Unemployment_Rate_Change_12M_Final,float64,712588,412,0.057784,946,True,REGULAR_PIT,FINAL_RESOLVED
94,BLS_Unemployment_Rate_Final,float64,712588,412,0.057784,260,True,REGULAR_PIT,FINAL_RESOLVED
50,BLS_Employment,float64,584343,128657,18.044460,55915,True,REGULAR_PIT,LOCAL_RAW


## 5. Define modeling roles and lineage

In [6]:
NON_PREDICTORS = {
    "Loan_ID", "Target_24M", "Split", "Dataset_Vintage",
    "Performance_Window_End", "COVID_Window_Overlap",
}

PIT_AUDIT_FIELDS = {
    "PIT_FHFA_Quarter_Date", "PIT_BLS_Month_Date",
    "Strict_PIT_FHFA_Quarter_Date", "Strict_PIT_BLS_Month_Date",
    "Strict_PIT_BLS_State_Month_Date", "BLS_State_Assumed_Available_Date",
    "Freddie_State", "Freddie_MSA", "Freddie_ZIP3",
    "LS_Freddie_State", "LS_Freddie_MSA", "LS_Freddie_ZIP3",
    "_State_FIPS", "09G_BLS_Remaining_Missing_Reason",
}

# FHFA / Housing lineage. Regular-PIT fields remain benchmark/audit lineage;
# LS_Housing_* are the final Strict-PIT housing candidates.
FHFA_INTERMEDIATE = {
    "FHFA_State_HPI", "FHFA_State_HPI_Growth_12M",
    "FHFA_ZIP3_HPI", "FHFA_ZIP3_HPI_Growth_12M",
    "FHFA_MSA_HPI", "FHFA_MSA_HPI_Growth_12M",
    "Housing_HPI", "Housing_HPI_Growth_12M",
    "Housing_Geography_Source", "Housing_Fallback_Level",
    "LS_FHFA_State_HPI", "LS_FHFA_State_HPI_Growth_12M",
    "LS_FHFA_ZIP3_HPI", "LS_FHFA_ZIP3_HPI_Growth_12M",
    "LS_FHFA_MSA_HPI", "LS_FHFA_MSA_HPI_Growth_12M",
}

STRICT_FINAL_HOUSING_NUMERIC = {
    "LS_Housing_HPI", "LS_Housing_HPI_Growth_12M",
}
STRICT_FINAL_HOUSING_PROVENANCE = {
    "LS_Housing_Geography_Source", "LS_Housing_Fallback_Level",
}

# Explicit old BLS components.
BLS_RAW_LOCAL = {
    "BLS_Unemployment_Rate", "BLS_Employment", "BLS_Labor_Force",
    "BLS_Employment_Growth_12M", "BLS_Unemployment_Rate_Change_12M",
    "Labor_Geography_Source",
    "LS_BLS_Unemployment_Rate", "LS_BLS_Employment", "LS_BLS_Labor_Force",
    "LS_BLS_Employment_Growth_12M", "LS_BLS_Unemployment_Rate_Change_12M",
    "LS_Labor_Geography_Source",
}

BLS_STATE_COMPONENTS = {
    "_State_BLS_Unemployment_Rate", "_State_BLS_Civilian_Labor_Force",
    "_State_BLS_Employment", "_State_BLS_Unemployment",
    "_State_BLS_Labor_Force_Participation_Rate",
    "_State_BLS_Employment_Population_Ratio",
    "LS_BLS_Unemployment_Rate_State_Fallback",
}

# Detect all final/resolved BLS fields from the NEW dataset.
# Strict/LS finals are production candidates.
# Non-LS finals are benchmark candidates only.
STRICT_BLS_FINAL = set()
REGULAR_BLS_FINAL = set()
BLS_FALLBACK_INTERMEDIATE_DYNAMIC = set()
BLS_PROVENANCE_DYNAMIC = set()

for c in bls_cols:
    u = c.upper()
    is_strict = c.startswith("LS_") or "STRICT" in u
    is_final = "FINAL" in u
    is_prov = any(k in u for k in ["GEOGRAPHY_SOURCE", "FALLBACK_LEVEL", "FALLBACK_APPLIED", "REMAINING_MISSING_REASON"])
    is_fallback_component = ("FALLBACK" in u and not is_final and not is_prov)

    if is_final and is_strict and not is_prov:
        STRICT_BLS_FINAL.add(c)
    elif is_final and (not is_strict) and not is_prov:
        REGULAR_BLS_FINAL.add(c)
    elif is_fallback_component:
        BLS_FALLBACK_INTERMEDIATE_DYNAMIC.add(c)
    elif is_prov:
        BLS_PROVENANCE_DYNAMIC.add(c)

# Ensure the known unemployment final is captured even in the legacy naming.
if "LS_BLS_Unemployment_Rate_Final" in df.columns:
    STRICT_BLS_FINAL.add("LS_BLS_Unemployment_Rate_Final")

print("Detected Strict/LS BLS final predictor fields:")
print(sorted(STRICT_BLS_FINAL))
print("\nDetected Regular-PIT BLS final benchmark fields:")
print(sorted(REGULAR_BLS_FINAL))
print("\nDetected BLS fallback/intermediate fields:")
print(sorted(BLS_FALLBACK_INTERMEDIATE_DYNAMIC))
print("\nDetected BLS provenance fields:")
print(sorted(BLS_PROVENANCE_DYNAMIC))

Detected Strict/LS BLS final predictor fields:
['LS_BLS_Employment_Growth_12M_Final', 'LS_BLS_Unemployment_Rate_Change_12M_Final', 'LS_BLS_Unemployment_Rate_Final']

Detected Regular-PIT BLS final benchmark fields:
['BLS_Employment_Growth_12M_Final', 'BLS_Unemployment_Rate_Change_12M_Final', 'BLS_Unemployment_Rate_Final']

Detected BLS fallback/intermediate fields:
['BLS_Employment_Growth_12M_State_Fallback', 'BLS_Unemployment_Rate_Change_12M_State_Fallback', 'BLS_Unemployment_Rate_State_Fallback', 'LS_BLS_Employment_Growth_12M_State_Fallback', 'LS_BLS_Unemployment_Rate_Change_12M_State_Fallback', 'LS_BLS_Unemployment_Rate_State_Fallback']

Detected BLS provenance fields:
['09G_BLS_Remaining_Missing_Reason', 'BLS_Fallback_Level', 'BLS_Geography_Source_Final', 'BLS_Remaining_Missing_Reason', 'BLS_State_Fallback_Applied', 'LS_BLS_Fallback_Level', 'LS_BLS_Geography_Source_Final', 'LS_BLS_State_Fallback_Applied', 'LS_Labor_Geography_Source', 'Labor_Geography_Source']


## 6. Assign a modeling role to every column

In [7]:
def infer_role(c):
    # IDs / target / split / audit
    if c == "Loan_ID":
        return "IDENTIFIER"
    if c == "Target_24M":
        return "TARGET"
    if c == "Split":
        return "SPLIT"
    if c in NON_PREDICTORS:
        return "AUDIT_METADATA"
    if c in PIT_AUDIT_FIELDS:
        return "PIT_OR_JOIN_AUDIT"

    # Special loan-level decisions from Notebook 03
    if c == "Original_DTI":
        return "LOAN_DTI_SPECIAL"
    if c == "Property_Valuation_Method":
        return "LOAN_DROP_AVAILABILITY_SHIFT"
    if c == "Special_Eligibility_Program":
        return "LOAN_HIGH_MISSING_CATEGORICAL_ABLATION"

    # Housing lineage
    if c in STRICT_FINAL_HOUSING_NUMERIC:
        return "STRICT_FINAL_MACRO_PREDICTOR"
    if c in STRICT_FINAL_HOUSING_PROVENANCE:
        return "STRICT_FINAL_MACRO_PROVENANCE"
    if c in FHFA_INTERMEDIATE:
        return "MACRO_INTERMEDIATE_OR_BENCHMARK"

    # BLS lineage — dynamic for new dataset
    if c in STRICT_BLS_FINAL:
        return "STRICT_FINAL_MACRO_PREDICTOR"
    if c in REGULAR_BLS_FINAL:
        return "REGULAR_PIT_FINAL_BENCHMARK"
    if c in BLS_RAW_LOCAL or c in BLS_STATE_COMPONENTS or c in BLS_FALLBACK_INTERMEDIATE_DYNAMIC:
        return "BLS_LINEAGE_INTERMEDIATE"
    if c in BLS_PROVENANCE_DYNAMIC:
        if c.startswith("LS_") and ("FINAL" in c.upper() or "FALLBACK_LEVEL" in c.upper() or "FALLBACK_APPLIED" in c.upper()):
            return "STRICT_FINAL_MACRO_PROVENANCE"
        return "BLS_LINEAGE_PROVENANCE_AUDIT"

    # Any remaining BLS column should not silently become a loan predictor.
    if "BLS" in c or "Labor_Geography" in c:
        return "BLS_UNCLASSIFIED_AUDIT"

    return "LOAN_LEVEL_PREDICTOR"

role_table = pd.DataFrame({
    "column": df.columns,
    "role": [infer_role(c) for c in df.columns],
})
display(role_table.groupby("role").size().rename("n_columns").reset_index())

,role,n_columns
0,AUDIT_METADATA,3
1,BLS_LINEAGE_INTERMEDIATE,24
2,BLS_LINEAGE_PROVENANCE_AUDIT,4
3,IDENTIFIER,1
4,LOAN_DROP_AVAILABILITY_SHIFT,1
5,LOAN_DTI_SPECIAL,1
6,LOAN_HIGH_MISSING_CATEGORICAL_ABLATION,1
7,LOAN_LEVEL_PREDICTOR,25
8,MACRO_INTERMEDIATE_OR_BENCHMARK,16
9,PIT_OR_JOIN_AUDIT,14


## 7. Freeze missing-value decisions

In [8]:
# Explicit loan-level numeric rules.
# Learned statistics are NOT calculated here; TRAIN_MEDIAN means the downstream preprocessing/modeling notebook
# must fit the median using Train rows only.
TRAIN_MEDIAN_NUMERIC = {
    "FICO_Score",
    "Number_of_Units",
    "Original_CLTV",
    "Original_LTV",
    "MI_Percentage",
}

def decision_for(c):
    role = infer_role(c)
    miss = int(df[c].isna().sum())
    dtype = df[c].dtype

    if role == "IDENTIFIER":
        return ("KEEP_AS_METADATA_NOT_PREDICTOR", "NONE",
                "Retain only for row identity and integrity checks.")

    if role == "TARGET":
        return ("KEEP_AS_TARGET_NOT_PREDICTOR", "NONE",
                "Outcome variable; never impute as a predictor.")

    if role == "SPLIT":
        return ("KEEP_AS_SPLIT_NOT_PREDICTOR", "NONE",
                "Controls Train/Validation/OOT partition; not a predictor.")

    if role in {"AUDIT_METADATA", "PIT_OR_JOIN_AUDIT"}:
        return ("KEEP_AS_AUDIT_NOT_PREDICTOR", "NONE",
                "Retain for temporal/merge audit, not for primary ML prediction.")

    if role in {"MACRO_INTERMEDIATE_OR_BENCHMARK", "BLS_LINEAGE_INTERMEDIATE"}:
        return ("DROP_FROM_PRIMARY_MODEL__KEEP_FOR_LINEAGE",
                "NO_IMPUTATION",
                "Raw/intermediate macro field is superseded by downstream resolved/final representation; do not independently impute.")

    if role == "BLS_LINEAGE_PROVENANCE_AUDIT":
        return ("KEEP_AS_AUDIT_NOT_PREDICTOR", "NO_IMPUTATION",
                "BLS merge/fallback provenance; retain for QA, not primary prediction.")

    if role == "BLS_UNCLASSIFIED_AUDIT":
        return ("REVIEW_REQUIRED", "UNRESOLVED",
                "New BLS-related column was not safely classified. Review its lineage before the downstream preprocessing/modeling notebook.")

    if role == "REGULAR_PIT_FINAL_BENCHMARK":
        if miss:
            return ("KEEP_BENCHMARK_ONLY_NOT_PRODUCTION",
                    "STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_BENCHMARK_PREPROCESSING",
                    "Final Regular-PIT BLS field is retained only for leakage-sensitivity benchmark; residual missingness must be handled explicitly later.")
        return ("KEEP_BENCHMARK_ONLY_NOT_PRODUCTION", "NO_TREATMENT",
                "Final Regular-PIT BLS field is benchmark-only; do not mix into the Strict/LS production feature set.")

    if role == "STRICT_FINAL_MACRO_PREDICTOR":
        if miss:
            return ("KEEP_PRIMARY_MACRO_CANDIDATE",
                    "STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_PREPROCESSING",
                    "Final Strict/LS resolved macro field. Preserve tiny/structural residual missingness for explicit downstream treatment; no global imputation here.")
        return ("KEEP_PRIMARY_MACRO_CANDIDATE", "NO_TREATMENT",
                "Final Strict/LS resolved macro field is complete.")

    if role == "STRICT_FINAL_MACRO_PROVENANCE":
        if miss:
            return ("KEEP_CANDIDATE_PROVENANCE", "EXPLICIT_MISSING_CATEGORY",
                    "Final Strict/LS provenance/fallback field; preserve missingness explicitly if used in an ablation.")
        return ("KEEP_CANDIDATE_PROVENANCE", "NO_TREATMENT",
                "Final Strict/LS provenance/fallback field; optional ablation feature, not core macro signal.")

    if role == "LOAN_DTI_SPECIAL":
        return ("KEEP_PRIMARY_PREDICTOR",
                "MISSING_INDICATOR_PLUS_TRAIN_MEDIAN",
                "Notebook 03 found strong Split/Vintage-driven missingness. Create missing indicator first; fit median on Train only; never use global median.")

    if role == "LOAN_DROP_AVAILABILITY_SHIFT":
        return ("DROP_FROM_PRIMARY_MODEL__OPTIONAL_ABLATION",
                "NO_IMPUTATION",
                "Severe temporal/Train-OOT availability mismatch; exclude from primary model rather than encoding data-collection regime.")

    if role == "LOAN_HIGH_MISSING_CATEGORICAL_ABLATION":
        return ("KEEP_CANDIDATE_ABLATION",
                "EXPLICIT_MISSING_CATEGORY",
                "Very high structural missingness; preserve missing as a category if tested, and compare via ablation.")

    if c in TRAIN_MEDIAN_NUMERIC:
        if miss:
            return ("KEEP_PRIMARY_PREDICTOR", "TRAIN_MEDIAN",
                    f"{miss} missing values; fit median using Train only and apply frozen value to all splits.")
        return ("KEEP_PRIMARY_PREDICTOR", "NO_TREATMENT", "No missing values.")

    # Dates should not be silently imputed or fed raw merely because complete.
    if pd.api.types.is_datetime64_any_dtype(dtype):
        if miss:
            return ("KEEP_FOR_LATER_FEATURE_ENGINEERING", "NO_IMPUTATION",
                    "Date field requires explicit feature engineering; do not silently impute in missing-value stage.")
        return ("KEEP_FOR_LATER_FEATURE_ENGINEERING", "NO_TREATMENT",
                "Complete date field; derive approved origination-time features later rather than feeding raw timestamp.")

    if miss == 0:
        return ("KEEP_PRIMARY_PREDICTOR", "NO_TREATMENT", "No missing values.")

    # Remaining categorical loan fields: explicit missing level, no mode imputation.
    if not pd.api.types.is_numeric_dtype(dtype):
        return ("KEEP_PRIMARY_PREDICTOR", "EXPLICIT_MISSING_CATEGORY",
                "Categorical missingness is preserved explicitly; no mode imputation.")

    # Numeric missing fields must be explicitly covered, never guessed.
    return ("REVIEW_REQUIRED", "UNRESOLVED",
            "Missing numeric predictor is not covered by an explicit frozen rule.")

rows = []
for c in df.columns:
    status, treatment, rationale = decision_for(c)
    rows.append({
        "column": c,
        "role": infer_role(c),
        "dtype": str(df[c].dtype),
        "missing_count": int(df[c].isna().sum()),
        "missing_pct": float(df[c].isna().mean() * 100),
        "unique_count": int(df[c].nunique(dropna=True)),
        "model_status": status,
        "missing_treatment": treatment,
        "rationale": rationale,
    })

decision_table = pd.DataFrame(rows)
display(decision_table)

,column,role,dtype,missing_count,missing_pct,unique_count,model_status,missing_treatment,rationale
0,Loan_ID,IDENTIFIER,str,0,0.000000,713000,KEEP_AS_METADATA_NOT_PREDICTOR,NONE,Retain only for row identity and integrity checks.
1,Dataset_Vintage,AUDIT_METADATA,float64,0,0.000000,17,KEEP_AS_AUDIT_NOT_PREDICTOR,NONE,"Retain for temporal/merge audit, not for primary ML prediction."
2,FICO_Score,LOAN_LEVEL_PREDICTOR,float64,100,0.014025,379,KEEP_PRIMARY_PREDICTOR,TRAIN_MEDIAN,100 missing values; fit median using Train only and apply frozen value to all splits.
3,First_Payment_Date,LOAN_LEVEL_PREDICTOR,datetime64[us],0,0.000000,194,KEEP_FOR_LATER_FEATURE_ENGINEERING,NO_TREATMENT,Complete date field; derive approved origination-time features later rather than feeding raw timestamp.
4,First_Time_Homebuyer,LOAN_LEVEL_PREDICTOR,string,13,0.001823,2,KEEP_PRIMARY_PREDICTOR,EXPLICIT_MISSING_CATEGORY,Categorical missingness is preserved explicitly; no mode imputation.
5,Maturity_Date,LOAN_LEVEL_PREDICTOR,datetime64[us],0,0.000000,441,KEEP_FOR_LATER_FEATURE_ENGINEERING,NO_TREATMENT,Complete date field; derive approved origination-time features later rather than feeding raw timestamp.
6,MSA,LOAN_LEVEL_PREDICTOR,string,89555,12.560309,455,KEEP_PRIMARY_PREDICTOR,EXPLICIT_MISSING_CATEGORY,Categorical missingness is preserved explicitly; no mode imputation.
7,MI_Percentage,LOAN_LEVEL_PREDICTOR,float64,3,0.000421,28,KEEP_PRIMARY_PREDICTOR,TRAIN_MEDIAN,3 missing values; fit median using Train only and apply frozen value to all splits.
8,Number_of_Units,LOAN_LEVEL_PREDICTOR,float64,62,0.008696,4,KEEP_PRIMARY_PREDICTOR,TRAIN_MEDIAN,62 missing values; fit median using Train only and apply frozen value to all splits.
9,Occupancy_Status,LOAN_LEVEL_PREDICTOR,string,0,0.000000,3,KEEP_PRIMARY_PREDICTOR,NO_TREATMENT,No missing values.


## 8. Hard QA gates

In [9]:
# 100% schema coverage, exactly once.
assert len(decision_table) == df.shape[1]
assert decision_table["column"].is_unique
assert set(decision_table["column"]) == set(df.columns)

# No row/key mutation.
assert len(df) == BASELINE_ROWS
assert df.shape[1] == BASELINE_COLS
assert df["Loan_ID"].nunique(dropna=False) == BASELINE_ROWS

# Known critical fields must retain intended decisions.
if "Original_DTI" in df.columns:
    r = decision_table.set_index("column").loc["Original_DTI"]
    assert r["missing_treatment"] == "MISSING_INDICATOR_PLUS_TRAIN_MEDIAN"

if "Property_Valuation_Method" in df.columns:
    r = decision_table.set_index("column").loc["Property_Valuation_Method"]
    assert str(r["model_status"]).startswith("DROP_FROM_PRIMARY_MODEL")

if "Special_Eligibility_Program" in df.columns:
    r = decision_table.set_index("column").loc["Special_Eligibility_Program"]
    assert r["missing_treatment"] == "EXPLICIT_MISSING_CATEGORY"

# Final LS unemployment must never be treated as an intermediate if present.
if "LS_BLS_Unemployment_Rate_Final" in df.columns:
    r = decision_table.set_index("column").loc["LS_BLS_Unemployment_Rate_Final"]
    assert r["role"] == "STRICT_FINAL_MACRO_PREDICTOR"

# Any detected Regular final BLS field must be benchmark-only.
for c in REGULAR_BLS_FINAL:
    r = decision_table.set_index("column").loc[c]
    assert r["model_status"] == "KEEP_BENCHMARK_ONLY_NOT_PRODUCTION"

# Any detected Strict/LS final BLS field must be a primary macro candidate.
for c in STRICT_BLS_FINAL:
    r = decision_table.set_index("column").loc[c]
    assert r["model_status"] == "KEEP_PRIMARY_MACRO_CANDIDATE"

review = decision_table.query("model_status == 'REVIEW_REQUIRED'")
if len(review):
    print("REVIEW_REQUIRED columns:")
    display(review)
    raise AssertionError(
        "At least one column is not covered by an explicit frozen rule. "
        "Review the displayed rows before exporting the manifest."
    )

print("All hard QA gates passed.")
print("Schema coverage:", f"{len(decision_table)}/{df.shape[1]}")

All hard QA gates passed.
Schema coverage: 105/105


## 9. Decision summaries

In [10]:
print("By role:")
display(decision_table.groupby("role").size().rename("n").reset_index().sort_values("n", ascending=False))

print("By model status:")
display(decision_table.groupby("model_status").size().rename("n").reset_index().sort_values("n", ascending=False))

print("By missing treatment:")
display(decision_table.groupby("missing_treatment").size().rename("n").reset_index().sort_values("n", ascending=False))

print("Final Strict/LS macro candidates:")
display(decision_table.query("model_status == 'KEEP_PRIMARY_MACRO_CANDIDATE'"))

print("Regular-PIT final benchmark candidates:")
display(decision_table.query("model_status == 'KEEP_BENCHMARK_ONLY_NOT_PRODUCTION'"))

By role:


,role,n
7,LOAN_LEVEL_PREDICTOR,25
1,BLS_LINEAGE_INTERMEDIATE,24
8,MACRO_INTERMEDIATE_OR_BENCHMARK,16
9,PIT_OR_JOIN_AUDIT,14
12,STRICT_FINAL_MACRO_PREDICTOR,5
13,STRICT_FINAL_MACRO_PROVENANCE,5
2,BLS_LINEAGE_PROVENANCE_AUDIT,4
0,AUDIT_METADATA,3
10,REGULAR_PIT_FINAL_BENCHMARK,3
4,LOAN_DROP_AVAILABILITY_SHIFT,1


By model status:


,model_status,n
0,DROP_FROM_PRIMARY_MODEL__KEEP_FOR_LINEAGE,40
11,KEEP_PRIMARY_PREDICTOR,24
2,KEEP_AS_AUDIT_NOT_PREDICTOR,21
8,KEEP_CANDIDATE_PROVENANCE,5
10,KEEP_PRIMARY_MACRO_CANDIDATE,5
6,KEEP_BENCHMARK_ONLY_NOT_PRODUCTION,3
9,KEEP_FOR_LATER_FEATURE_ENGINEERING,2
1,DROP_FROM_PRIMARY_MODEL__OPTIONAL_ABLATION,1
3,KEEP_AS_METADATA_NOT_PREDICTOR,1
7,KEEP_CANDIDATE_ABLATION,1


By missing treatment:


,missing_treatment,n
3,NO_IMPUTATION,45
4,NO_TREATMENT,21
2,NONE,20
0,EXPLICIT_MISSING_CATEGORY,5
7,TRAIN_MEDIAN,5
6,STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_PREPROCESSING,5
5,STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_BENCHMARK_PREPROCESSING,3
1,MISSING_INDICATOR_PLUS_TRAIN_MEDIAN,1


Final Strict/LS macro candidates:


,column,role,dtype,missing_count,missing_pct,unique_count,model_status,missing_treatment,rationale
66,LS_Housing_HPI,STRICT_FINAL_MACRO_PREDICTOR,float64,329,0.046143,20228,KEEP_PRIMARY_MACRO_CANDIDATE,STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_PREPROCESSING,Final Strict/LS resolved macro field. Preserve tiny/structural residual missingness for explicit downstream treatment; no global imputation here.
67,LS_Housing_HPI_Growth_12M,STRICT_FINAL_MACRO_PREDICTOR,float64,329,0.046143,48699,KEEP_PRIMARY_MACRO_CANDIDATE,STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_PREPROCESSING,Final Strict/LS resolved macro field. Preserve tiny/structural residual missingness for explicit downstream treatment; no global imputation here.
86,LS_BLS_Unemployment_Rate_Final,STRICT_FINAL_MACRO_PREDICTOR,Float64,333,0.046704,261,KEEP_PRIMARY_MACRO_CANDIDATE,STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_PREPROCESSING,Final Strict/LS resolved macro field. Preserve tiny/structural residual missingness for explicit downstream treatment; no global imputation here.
103,LS_BLS_Employment_Growth_12M_Final,STRICT_FINAL_MACRO_PREDICTOR,float64,333,0.046704,69481,KEEP_PRIMARY_MACRO_CANDIDATE,STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_PREPROCESSING,Final Strict/LS resolved macro field. Preserve tiny/structural residual missingness for explicit downstream treatment; no global imputation here.
104,LS_BLS_Unemployment_Rate_Change_12M_Final,STRICT_FINAL_MACRO_PREDICTOR,float64,333,0.046704,963,KEEP_PRIMARY_MACRO_CANDIDATE,STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_PREPROCESSING,Final Strict/LS resolved macro field. Preserve tiny/structural residual missingness for explicit downstream treatment; no global imputation here.


Regular-PIT final benchmark candidates:


,column,role,dtype,missing_count,missing_pct,unique_count,model_status,missing_treatment,rationale
94,BLS_Unemployment_Rate_Final,REGULAR_PIT_FINAL_BENCHMARK,float64,412,0.057784,260,KEEP_BENCHMARK_ONLY_NOT_PRODUCTION,STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_BENCHMARK_PREPROCESSING,Final Regular-PIT BLS field is retained only for leakage-sensitivity benchmark; residual missingness must be handled explicitly later.
95,BLS_Employment_Growth_12M_Final,REGULAR_PIT_FINAL_BENCHMARK,float64,412,0.057784,69478,KEEP_BENCHMARK_ONLY_NOT_PRODUCTION,STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_BENCHMARK_PREPROCESSING,Final Regular-PIT BLS field is retained only for leakage-sensitivity benchmark; residual missingness must be handled explicitly later.
96,BLS_Unemployment_Rate_Change_12M_Final,REGULAR_PIT_FINAL_BENCHMARK,float64,412,0.057784,946,KEEP_BENCHMARK_ONLY_NOT_PRODUCTION,STRUCTURAL_RESIDUAL_MISSING__DEFER_TO_BENCHMARK_PREPROCESSING,Final Regular-PIT BLS field is retained only for leakage-sensitivity benchmark; residual missingness must be handled explicitly later.


## 10. Freeze the updated manifest for the downstream preprocessing/modeling notebook

In [11]:
# Resolve the project root robustly.
# Expected project:
# C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 2
#
# Whether this notebook is run from the project root or from /notebooks,
# all outputs below are written under PROJECT_ROOT/data/output.

cwd = Path.cwd().resolve()

if (cwd / "data").is_dir():
    PROJECT_ROOT = cwd
elif cwd.name.lower() == "notebooks" and (cwd.parent / "data").is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd.parent / "data").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not resolve the project root containing the 'data' directory. "
        f"Current working directory: {cwd}"
    )

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / "missing_value_decisions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "03_updated_final_missing_value_decision_table.csv"
json_path = OUTPUT_DIR / "03_updated_final_missing_value_decision_manifest.json"
mask_path = OUTPUT_DIR / "03_updated_identical_missing_mask_groups.csv"
baseline_path = OUTPUT_DIR / "03_updated_preprocessing_baseline_fingerprint.json"
bls_path = OUTPUT_DIR / "03_updated_BLS_lineage_inventory.csv"

decision_table.to_csv(csv_path, index=False)
mask_group_table.to_csv(mask_path, index=False)
bls_inventory.to_csv(bls_path, index=False)

manifest = {
    "source_artifact": FILE_NAME,
    "observed_shape": [int(df.shape[0]), int(df.shape[1])],
    "expected_rows": 713000,
    "column_count_is_dynamic": True,
    "stage": "missing_value_decisions_frozen_pre_imputation_updated_BLS",
    "feature_engineering_performed": False,
    "imputation_performed": False,
    "strict_ls_bls_final_fields_detected": sorted(STRICT_BLS_FINAL),
    "regular_pit_bls_final_fields_detected": sorted(REGULAR_BLS_FINAL),
    "rules": decision_table.to_dict(orient="records"),
}
json_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
baseline_path.write_text(
    json.dumps(baseline, indent=2),
    encoding="utf-8"
)

print("Project root :", PROJECT_ROOT)
print("Output dir   :", OUTPUT_DIR)
print("\nSaved:")
for p in [csv_path, json_path, mask_path, baseline_path, bls_path]:
    print(" -", p.resolve())

Project root : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6
Output dir   : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\missing_value_decisions

Saved:
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\missing_value_decisions\03_updated_final_missing_value_decision_table.csv
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\missing_value_decisions\03_updated_final_missing_value_decision_manifest.json
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\missing_value_decisions\03_updated_identical_missing_mask_groups.csv
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\missing_value_decisions\03_updated_preprocessing_baseline_fingerprint.json
 - C:\Users\itwill\Desktop\study\코딩공

## 11. Hand-off to the downstream preprocessing/modeling notebook

the downstream preprocessing/modeling notebook should now consume the **updated manifest**, not re-decide missing-value policy.

The intended model-set logic remains:

- **Loan-only baseline:** selected origination-time loan predictors.
- **Production macro model:** selected loan predictors + **Strict/LS final-resolved macro predictors**.
- **Regular-PIT comparison:** same loan predictors + **Regular-PIT final-resolved macro benchmark fields**, used only to quantify the effect of looser PIT construction.
- raw MSA/state/fallback construction fields remain lineage/audit fields and should not be independently imputed and simultaneously fed alongside their final resolved descendants.

Any learned imputation statistic must be fit on **Train only**.